In [ ]:
# Before running install these packages:
# pip install torchcodec opencv-python matplotlib

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

# =========================
# 1. Paths and model config
# =========================

model_path = "Alibaba-DAMO-Academy/RynnBrain-2B"

cache_dir = r"E:\Master - Polimi\Thesis\Practical steps\RynnBrain\hf_cache"

video_path = r"E:\Master - Polimi\Thesis\Practical steps\data\videos\your_video.mp4"


# =========================
# 2. Check GPU
# =========================

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    dtype = torch.float16
else:
    print("Running on CPU")
    dtype = torch.float32


# =========================
# 3. Load processor and model
# =========================

processor = AutoProcessor.from_pretrained(
    model_path,
    cache_dir=cache_dir
)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    cache_dir=cache_dir,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()


# =========================
# 4. Video prompt
# =========================

conversation = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": video_path
            },
            {
                "type": "text",
                "text": (
                    "This video shows a robotic manipulation task. "
                    "Describe what happens in the video, focusing on the robot, "
                    "the manipulated object, and the final state of the task."
                )
            },
        ],
    }
]


# =========================
# 5. Inference
# =========================

model_inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

model_inputs = model_inputs.to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
    )

output_ids = output_ids[:, model_inputs["input_ids"].shape[1]:]

response = processor.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(response)